In [1]:
import pandas as pd, geopandas as gpd, duckdb, statsmodels, sklearn, shap
print("All good")

Fontconfig warning: ignoring UTF-8: not a valid region tag
Matplotlib is building the font cache; this may take a moment.


All good


In [2]:
from pathlib import Path
import geopandas as gpd

In [3]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

In [4]:
health_regions = gpd.read_file(RAW / "boundaries" / "HR_000b25a_e" / "HR_000b25a_e.shp")
print(health_regions.shape)
print(health_regions.columns.tolist())

(91, 5)
['HR_UID', 'ENGNAME', 'FRENAME', 'DGUID', 'geometry']


In [5]:
# has 91 health regions
health_regions.drop(columns="geometry").head(10) # drop geometry to make it readable

,HR_UID,ENGNAME,FRENAME,DGUID
0,1020,Eastern Urban Zone,Eastern Urban Zone,2025A00071020
1,1021,Eastern Rural Zone,Eastern Rural Zone,2025A00071021
2,1022,Central Zone,Central Zone,2025A00071022
3,1023,Western Zone,Western Zone,2025A00071023
4,1024,Labrador-Grenfell Zone,Labrador-Grenfell Zone,2025A00071024
5,1100,Prince Edward Island,Île-du-Prince-Édouard,2025A00071100
6,1201,Zone 1 - Western,Zone 1 - Western,2025A00071201
7,1202,Zone 2 - Northern,Zone 2 - Northern,2025A00071202
8,1203,Zone 3 - Eastern,Zone 3 - Eastern,2025A00071203
9,1204,Zone 4 - Central,Zone 4 - Central,2025A00071204


The first two digits are the provinces- 10 = Newfoundland and Labrador, 11 = PEI, 12 = Nova Scotia, 48 = Alberta

In [8]:
alberta = health_regions[health_regions["HR_UID"].astype(str).str.startswith("48")]
alberta.drop(columns="geometry")

,HR_UID,ENGNAME,FRENAME,DGUID
76,4841,Southwest Corridor,Southwest Corridor,2025A00074841
77,4842,Southeast Corridor,Southeast Corridor,2025A00074842
78,4843,Calgary Corridor,Calgary Corridor,2025A00074843
79,4844,Central Corridor,Central Corridor,2025A00074844
80,4845,Edmonton Corridor,Edmonton Corridor,2025A00074845
81,4846,Northwest Corridor,Northwest Corridor,2025A00074846
82,4847,Northeast Corridor,Northeast Corridor,2025A00074847


In [9]:
health_regions["province_code"] = health_regions["HR_UID"].astype(str).str[:2]
health_regions.groupby("province_code").size()

province_code
10     5
11     1
12     4
13     7
24    18
35    29
46     5
47     7
48     7
59     5
60     1
61     1
62     1
dtype: int64

Ontario (35) has the most regions- 29 which is 32% of the total regions. It can dominate clustering and regression if it had data that behaved differently than the rest of the country. We will watch out for that. Some regions are huge such as the PEI (11) so it might behave as an outlier.

In [10]:
health_regions[health_regions["province_code"] == "35"][["HR_UID", "ENGNAME"]].head(10)

,HR_UID,ENGNAME
35,3526,The District of Algoma Health Unit
36,3530,Durham Regional Health Unit
37,3533,Grey Bruce Health Unit
38,3536,Halton Regional Health Unit
39,3537,City of Hamilton Health Unit
40,3540,Chatham-Kent Health Unit
41,3542,Lambton Health Unit
42,3544,Middlesex-London Health Unit
43,3546,Niagara Regional Area Health Unit
44,3547,North Bay Parry Sound District Health Unit
